# Stage 2 shard packing on a Colab **CPU** runtime (Hugging Face only, no Google Drive)

Packs every Stage 2 shard in Google's datacenter and uploads it straight to the private dataset repo
`Wosca/math-recogniser-shards`, so nothing large ever crosses a home connection.

1. **Runtime → Change runtime type → CPU.** High-RAM CPU is recommended (8+ vCPUs, ~30 min end to end, < 0.1 CU);
   the free standard CPU runtime (0 CU) also works, just slower.
2. Add a **write**-scoped Hugging Face token as **`HF_TOKEN`** in the left sidebar **🔑 Secrets** (toggle
   **Notebook access** ON).
3. Click **Runtime → Run all**.

What it does: pulls `math-recogniser.zip` and the small `seed/stage2_seed.tar.gz` (CROHME 2023 jsonl, structure
annotations, HME100K `examples.jsonl`, the locked Stage 1 `tokenizer.json`, `manifest_seed.json` and the packed
`val_dev` shard) from `HF_SHARDS_REPO`; downloads MathWriting 2024 (GCS), HME100K (2 public parquets, rebuilt
byte-for-byte by SHA-256) and Detexify; packs `stage2_train` with the full test-set leakage audit and the **exact**
Stage 1 vocabulary (`--fixed-tokenizer`); gzip-1 compresses the shards; uploads `shards/stage2_train/shard-*.tar.gz`,
`shards/manifest.json`, `shards/tokenizer.json`, `shards/audit_report.json` and `shards/val_dev/` with
`upload_large_folder`; verifies the repo listing; then disconnects the runtime.

Every step is resumable: rerunning skips finished downloads, already packed shards and already uploaded files.

In [ ]:
# 1. Configuration, secrets and runtime check
HF_SHARDS_REPO = "Wosca/math-recogniser-shards"
AUTO_DISCONNECT = True             # release the runtime once the upload is verified
import os

WORKERS = max(2, min(12, os.cpu_count() or 2))
AST_COUNT = 30000
EXPECTED_STAGE2_SHARDS = 138       # reference count from the local pack; a different count only warns
LOCKED_TOKENIZER_SIZE = 907        # Stage 1 vocabulary; packing aborts if the seed tokenizer differs
MIN_FREE_GIB = 25                  # below this, each .tar is deleted once its verified .tar.gz exists

import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

CONTENT = Path("/content")
REPO_DIR = CONTENT / "math-recogniser"
ML_DIR = REPO_DIR / "ml"
SEED_DIR = CONTENT / "seed"
DATA_DIR = CONTENT / "data"
OUTPUT = CONTENT / "output"
UPLOAD = CONTENT / "upload"
UPLOAD_SHARDS = UPLOAD / "shards"

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"   # public downloads stay anonymous; HF_TOKEN only for HF_SHARDS_REPO
HF_TOKEN = ""
try:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception as error:
    print(f"Colab Secrets: {type(error).__name__}: {error}")
HF_TOKEN = (HF_TOKEN or os.environ.get("HF_TOKEN", "")).strip()
if not HF_TOKEN.startswith("hf_"):
    raise SystemExit("HF_TOKEN missing: add a write-scoped Hugging Face token named HF_TOKEN in the left sidebar "
                     "Secrets (key icon) and switch Notebook access ON, then rerun.")
if HF_SHARDS_REPO.count("/") != 1 or not all(HF_SHARDS_REPO.split("/")):
    raise SystemExit('HF_SHARDS_REPO must look like "<user>/<repo>"')

ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30 if hasattr(os, "sysconf") else 0.0
free_gib = shutil.disk_usage(CONTENT).free / 2**30
print(f"CPUs {os.cpu_count()} | RAM {ram_gib:.1f} GiB | free disk on /content {free_gib:.1f} GiB | "
      f"pack workers {WORKERS}")
if free_gib < 60:
    print(f"warning: only {free_gib:.0f} GiB free; MathWriting + Stage 2 shards need roughly 40-60 GiB")

In [ ]:
# 2. Lightweight dependencies, repository and seed bundle from HF_SHARDS_REPO (PyTorch is never touched)
import gzip
import tarfile
import zipfile


def sh(*command, cwd=None):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], check=True, cwd=cwd)


sh(sys.executable, "-m", "pip", "install", "-q", "pyyaml", "antlr4-python3-runtime>=4.11,<4.12", "pyarrow",
   "huggingface_hub>=0.24")
from huggingface_hub import HfApi, hf_hub_download

STAGING = CONTENT / "_hf"


def fetch_private(filename: str) -> Path:
    return Path(hf_hub_download(repo_id=HF_SHARDS_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN,
                                local_dir=STAGING))


def decompress(archive: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".tmp")
    with gzip.open(archive, "rb") as source, open(temporary, "wb") as target:
        shutil.copyfileobj(source, target, length=1 << 22)
    os.replace(temporary, destination)


zip_path = fetch_private("math-recogniser.zip")
seed_path = fetch_private("seed/stage2_seed.tar.gz")
print(f"math-recogniser.zip {zip_path.stat().st_size:,} B | stage2_seed.tar.gz {seed_path.stat().st_size:,} B")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
unpacked = CONTENT / "_repo_unpack"
shutil.rmtree(unpacked, ignore_errors=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(unpacked)
roots = [path for path in unpacked.iterdir() if path.is_dir() and (path / "ml").is_dir()]
shutil.move(str(roots[0] if roots else unpacked), str(REPO_DIR))
shutil.rmtree(unpacked, ignore_errors=True)
PACKER = ML_DIR / "data" / "bimodal_pack.py"
if not PACKER.is_file():
    raise SystemExit(f"{PACKER} missing from math-recogniser.zip")
if '"--fixed-tokenizer"' not in PACKER.read_text(encoding="utf-8"):
    raise SystemExit("math-recogniser.zip predates --fixed-tokenizer: re-upload the zip, then rerun.")
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

shutil.rmtree(SEED_DIR, ignore_errors=True)
SEED_DIR.mkdir(parents=True)
with tarfile.open(seed_path, "r:gz") as archive:
    archive.extractall(SEED_DIR, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))
nested = SEED_DIR / "seed"
if nested.is_dir() and not (SEED_DIR / "shards").exists():
    for child in list(nested.iterdir()):
        shutil.move(str(child), str(SEED_DIR / child.name))
    nested.rmdir()
SEED_FILES = ["crohme23/train.jsonl", "crohme23/validation.jsonl", "crohme23/test.jsonl", "crohme23/manifest.json",
              "crohme-all-train-v2.json", "hme100k/examples.jsonl", "shards/tokenizer.json",
              "shards/manifest_seed.json", "shards/val_dev/shard-00000.tar.gz"]
missing = [name for name in SEED_FILES if not (SEED_DIR / name).is_file()]
if missing:
    raise SystemExit(f"stage2_seed.tar.gz is missing {missing}")
LOCKED_TOKENIZER = SEED_DIR / "shards" / "tokenizer.json"
locked_size = len(json.loads(LOCKED_TOKENIZER.read_text(encoding="utf-8"))["tokens"])
if locked_size != LOCKED_TOKENIZER_SIZE:
    raise SystemExit(f"seed tokenizer.json has {locked_size} tokens, expected the locked {LOCKED_TOKENIZER_SIZE}")

OUTPUT.mkdir(parents=True, exist_ok=True)
for archive in sorted((SEED_DIR / "shards" / "val_dev").glob("shard-*.tar.gz")):
    decompress(archive, OUTPUT / "val_dev" / archive.name[:-3])
shutil.copyfile(SEED_DIR / "shards" / "manifest_seed.json", OUTPUT / "manifest.json")
seed_manifest = json.loads((OUTPUT / "manifest.json").read_text(encoding="utf-8"))
if "val" not in seed_manifest.get("stages", {}):
    raise SystemExit("manifest_seed.json has no val stage")
shutil.rmtree(STAGING, ignore_errors=True)
print(f"repo -> {REPO_DIR} | seed -> {SEED_DIR} | locked tokenizer {locked_size} tokens | "
      f"seed stages {sorted(seed_manifest['stages'])}")

In [ ]:
# 3. Public datasets at datacenter speed: MathWriting 2024 (GCS), HME100K (2 parquets), Detexify
import hashlib

import pyarrow.parquet as pq

DATA_DIR.mkdir(parents=True, exist_ok=True)
started = time.time()

# MathWriting 2024
MW_URL = "https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz"
MW_TGZ = DATA_DIR / "mathwriting-2024.tgz"
MW_ROOT = DATA_DIR / "mathwriting-2024"
MW_DONE = MW_ROOT / ".extracted"
if not MW_DONE.is_file():
    sh("curl", "-L", "--fail", "--retry", "5", "--retry-delay", "5", "-C", "-", "-o", MW_TGZ, MW_URL)
    sh("tar", "-xzf", MW_TGZ, "-C", DATA_DIR)
    if not (MW_ROOT / "train").is_dir():
        found = [path.parent for path in DATA_DIR.glob("*/train") if (path.parent / "test").is_dir()]
        if not found:
            raise SystemExit(f"{MW_TGZ.name} did not contain train/ and test/")
        MW_ROOT = found[0]
        MW_DONE = MW_ROOT / ".extracted"
    MW_DONE.write_text("ok", encoding="utf-8")
    MW_TGZ.unlink(missing_ok=True)
for split in ("train", "valid", "test", "synthetic", "symbols"):
    if not (MW_ROOT / split).is_dir():
        raise SystemExit(f"MathWriting split {split}/ missing under {MW_ROOT}")
print(f"MathWriting -> {MW_ROOT} ({time.time() - started:.0f}s)", flush=True)

# HME100K: rebuild images/<sha256>.img byte-for-byte from the pinned public mirror
HME_ROOT = SEED_DIR / "hme100k"
HME_REPO = "ConstantHao/HME100K"
HME_REVISION = "68ea96bd2527aac3799f94803a5f86b92fd9c2d2"
HME_PARQUETS = ["data/train-00000-of-00002.parquet", "data/train-00001-of-00002.parquet"]
wanted: dict[str, Path] = {}
with open(HME_ROOT / "examples.jsonl", "r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            row = json.loads(line)
            wanted[row["sha256"]] = HME_ROOT / row["image"]
pending = {digest: path for digest, path in wanted.items() if not path.is_file()}
print(f"HME100K: {len(wanted)} images referenced, {len(pending)} to rebuild", flush=True)
if pending:
    parquet_dir = DATA_DIR / "hme100k_parquet"
    for name in HME_PARQUETS:
        local = Path(hf_hub_download(repo_id=HME_REPO, filename=name, repo_type="dataset", revision=HME_REVISION,
                                     token=False, local_dir=parquet_dir))
        rows = matched = 0
        for batch in pq.ParquetFile(local).iter_batches(batch_size=512, columns=["image"]):
            for cell in batch.column("image").to_pylist():
                rows += 1
                data = cell.get("bytes") if isinstance(cell, dict) else cell
                if not data:
                    continue
                target = pending.pop(hashlib.sha256(data).hexdigest(), None)
                if target is None:
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = target.with_name(target.name + ".tmp")
                temporary.write_bytes(data)
                os.replace(temporary, target)
                matched += 1
        local.unlink(missing_ok=True)
        print(f"  {name}: {rows} rows, {matched} images written", flush=True)
    shutil.rmtree(parquet_dir, ignore_errors=True)
present = sum(path.is_file() for path in wanted.values())
print(f"HME100K: {present}/{len(wanted)} images present ({time.time() - started:.0f}s)", flush=True)
if present != len(wanted):
    raise SystemExit(f"HME100K: {len(wanted) - present} referenced images are not in {HME_REPO}@{HME_REVISION}")

# Detexify (public Google Drive files, parsed into samples.jsonl)
EXTERNAL = DATA_DIR / "external"
if not (EXTERNAL / "detexify" / "samples.jsonl").is_file():
    sh(sys.executable, "-u", "-m", "data.bimodal_pack", "--download-detexify", "--external-root", EXTERNAL, cwd=ML_DIR)
print(f"Detexify -> {EXTERNAL / 'detexify'} | downloads done in {time.time() - started:.0f}s "
      f"| free disk {shutil.disk_usage(CONTENT).free / 2**30:.1f} GiB", flush=True)

In [ ]:
# 4. Pack Stage 2 (leakage audit + locked Stage 1 tokenizer), verify SHA-256, gzip-1 compress in parallel
from concurrent.futures import ThreadPoolExecutor

from data.bimodal_shards import sha256_file, verify_manifest_checksums

command = [sys.executable, "-u", "-m", "data.bimodal_pack", "--output", OUTPUT, "--stages", "stage2",
           "--workers", WORKERS, "--shard-size", "5000", "--max-tokens", "256",
           "--crohme-root", SEED_DIR / "crohme23", "--mathwriting-root", MW_ROOT, "--hme-root", HME_ROOT,
           "--structure", SEED_DIR / "crohme-all-train-v2.json", "--external-root", EXTERNAL, "--with-detexify",
           "--ast-count", AST_COUNT, "--crohme-upsample", "3.0", "--stage2-hme-fraction", "0.3",
           "--block-test-writers", "--fixed-tokenizer", LOCKED_TOKENIZER]
command = [str(part) for part in command]
print("$", " ".join(command), flush=True)
started = time.time()
process = subprocess.Popen(command, cwd=ML_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                           bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
for line in process.stdout:
    print(line, end="")
if process.wait() != 0:
    raise SystemExit(f"bimodal_pack exited with code {process.returncode}")
print(f"packed in {time.time() - started:.0f}s", flush=True)

manifest = json.loads((OUTPUT / "manifest.json").read_text(encoding="utf-8"))
if not {"val", "stage2"} <= set(manifest["stages"]):
    raise SystemExit(f"manifest.json stages {sorted(manifest['stages'])} lack val + stage2")
if not manifest["tokenizer"].get("fixed_tokenizer") or sha256_file(OUTPUT / "tokenizer.json") != sha256_file(
        LOCKED_TOKENIZER):
    raise SystemExit("tokenizer.json is not the locked Stage 1 tokenizer")
if manifest["tokenizer"].get("oov_labels"):
    print(f"warning: {manifest['tokenizer']['oov_labels']} Stage 2 labels use tokens outside the Stage 1 vocabulary "
          f"{manifest['tokenizer'].get('oov_tokens')}")
expected = {shard["file"]: shard["sha256"] for shard in manifest["stages"]["stage2"]["shards"]}
checks = {key: ok for key, ok in verify_manifest_checksums(OUTPUT).items()
          if key.startswith("val_dev/") or (key.startswith("stage2_train/")
                                            and (OUTPUT / key).is_file())}
bad = [key for key, ok in checks.items() if not ok]
if bad:
    raise SystemExit(f"Checksum mismatch: {bad[:10]}")
print(f"verified {len(checks)} shard checksum(s); stage2: {manifest['stages']['stage2']['samples']} samples in "
      f"{len(expected)} shard(s) {manifest['stages']['stage2']['sources']}")
if len(expected) != EXPECTED_STAGE2_SHARDS:
    print(f"warning: {len(expected)} stage2 shards, expected {EXPECTED_STAGE2_SHARDS} from the local reference pack")


def gz_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with gzip.open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 22), b""):
            digest.update(chunk)
    return digest.hexdigest()


def compress(name: str) -> tuple[str, bool, int]:
    source = OUTPUT / "stage2_train" / name
    target = UPLOAD_SHARDS / "stage2_train" / f"{name}.gz"
    if target.is_file() and not source.is_file():
        return name, gz_sha256(target) == expected[name], target.stat().st_size
    temporary = target.with_name(target.name + ".tmp")
    with open(source, "rb") as raw, open(temporary, "wb") as sink, \
            gzip.GzipFile(filename="", mode="wb", compresslevel=1, fileobj=sink, mtime=0) as packed:
        shutil.copyfileobj(raw, packed, length=1 << 22)
    if gz_sha256(temporary) != expected[name]:
        temporary.unlink(missing_ok=True)
        return name, False, 0
    os.replace(temporary, target)
    if shutil.disk_usage(CONTENT).free < MIN_FREE_GIB * 2**30:
        source.unlink()
    return name, True, target.stat().st_size


(UPLOAD_SHARDS / "stage2_train").mkdir(parents=True, exist_ok=True)
(UPLOAD_SHARDS / "val_dev").mkdir(parents=True, exist_ok=True)
for stale in (UPLOAD_SHARDS / "stage2_train").iterdir():
    if stale.name.endswith(".tmp") or stale.name[:-3] not in expected:
        stale.unlink()
missing = [name for name in expected if not (OUTPUT / "stage2_train" / name).is_file()
           and not (UPLOAD_SHARDS / "stage2_train" / f"{name}.gz").is_file()]
if missing:
    raise SystemExit(f"stage2 shards missing locally: {missing[:10]}")
started = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    results = list(pool.map(compress, sorted(expected)))
failed = [name for name, ok, _ in results if not ok]
if failed:
    raise SystemExit(f"gzip verification failed for {failed[:10]}")
for name in ("manifest.json", "tokenizer.json", "audit_report.json"):
    shutil.copyfile(OUTPUT / name, UPLOAD_SHARDS / name)
for archive in sorted((SEED_DIR / "shards" / "val_dev").glob("shard-*.tar.gz")):
    shutil.copyfile(archive, UPLOAD_SHARDS / "val_dev" / archive.name)
compressed = sum(size for _, _, size in results)
print(f"gzip-1: {len(results)} shard(s), {compressed / 2**30:.2f} GiB in {time.time() - started:.0f}s "
      f"-> {UPLOAD_SHARDS}", flush=True)

In [ ]:
# 5. Upload to the private HF dataset repo (multi-commit, resumable), verify the listing, auto-disconnect
from huggingface_hub import CommitOperationDelete

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_SHARDS_REPO, repo_type="dataset", private=True, exist_ok=True)
started = time.time()
api.upload_large_folder(repo_id=HF_SHARDS_REPO, repo_type="dataset", folder_path=str(UPLOAD),
                        ignore_patterns=["*.tmp"], num_workers=min(16, max(4, WORKERS)))
print(f"upload_large_folder finished in {time.time() - started:.0f}s", flush=True)

required = {"shards/manifest.json", "shards/tokenizer.json", "shards/audit_report.json",
            *(f"shards/val_dev/{path.name}" for path in (UPLOAD_SHARDS / "val_dev").glob("shard-*.tar.gz")),
            *(f"shards/stage2_train/{name}.gz" for name in expected)}
remote = set(api.list_repo_files(HF_SHARDS_REPO, repo_type="dataset"))
absent = sorted(required - remote)
if absent:
    raise SystemExit(f"{len(absent)} file(s) missing from {HF_SHARDS_REPO} after upload: {absent[:10]}; rerun this cell")
stale = sorted(path for path in remote if path.startswith("shards/stage2_train/shard-") and path not in required)
if stale:
    api.create_commit(repo_id=HF_SHARDS_REPO, repo_type="dataset",
                      operations=[CommitOperationDelete(path_in_repo=path) for path in stale],
                      commit_message=f"stage2: remove {len(stale)} stale shard(s)")
    print(f"removed {len(stale)} stale stage2 shard(s) from {HF_SHARDS_REPO}")
stage2_remote = sorted(path for path in required if path.startswith("shards/stage2_train/"))
print(f"DONE: {len(stage2_remote)} stage2_train shards + manifest.json + tokenizer.json + audit_report.json + val_dev "
      f"verified on https://huggingface.co/datasets/{HF_SHARDS_REPO}/tree/main/shards")
print("Next: open colab/train_bimodal_g4.ipynb with STORAGE_MODE = \"hf\" and STAGE = 2.")
if AUTO_DISCONNECT:
    print("Disconnecting the runtime in 15 seconds (set AUTO_DISCONNECT = False to keep it)...", flush=True)
    time.sleep(15)
    from google.colab import runtime

    runtime.unassign()